# 🚛 ANN-Based Indian Truck Trip Cost Prediction

## Project Introduction

This notebook implements an end-to-end **Artificial Neural Network (ANN) regression model** to predict the operating cost of truck trips across Indian routes.

### Business Problem
A transport operator needs to estimate trip costs before dispatch to plan budgets, set freight rates, and manage fleet expenses. This model predicts `actual_trip_cost_inr` based on route, vehicle, fuel, and cost parameters.

### ⚠️ Synthetic Data Disclosure
**This dataset is SYNTHETIC and generated for educational purposes only.** It does not represent actual company data, Delhivery data, or government logistics statistics. The target variable (`actual_trip_cost_inr`) was formula-generated, so high evaluation metrics reflect this deterministic formula rather than production-grade accuracy on real-world data.

### Approach
1. Data loading and quality checks
2. Exploratory Data Analysis (EDA)
3. Feature engineering and preprocessing
4. ANN model training with early stopping
5. Evaluation with MAE, RMSE, R² and visual diagnostics
6. Comparison against baseline models
7. Artifact saving for deployment

In [ ]:
# Standard imports
import os
import sys
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

# Project imports
sys.path.insert(0, os.path.abspath('..'))
from src.data_utils import (
    load_dataset, validate_columns, data_quality_report,
    separate_features_target, build_preprocessor, get_feature_names,
    build_baseline, NUMERICAL_FEATURES, CATEGORICAL_FEATURES,
    ALL_FEATURE_COLUMNS, TARGET_COLUMN, DROP_COLUMNS
)

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# Plot style
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

print(f'Python:      {sys.version}')
print(f'TensorFlow:  {tf.__version__}')
print(f'Pandas:      {pd.__version__}')
print(f'NumPy:       {np.__version__}')

---
## 1. Dataset Loading and Structure

In [ ]:
# Load dataset
DATA_PATH = os.path.join('..', 'data', 'indian_truck_logistics_synthetic_15000.csv')
df = load_dataset(DATA_PATH)

print(f'Dataset shape: {df.shape}')
print(f'Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}')
df.head()

In [ ]:
# Column info
df.info()

In [ ]:
# Validate columns
validate_columns(df)
print('✅ All 23 required columns are present.')

---
## 2. Data Quality Checks

In [ ]:
# Missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.sum() > 0 else 'None — all columns are complete.')
print(f'\nTotal missing values: {missing.sum()}')

In [ ]:
# Duplicate rows
dupes = df.duplicated().sum()
print(f'Duplicate rows: {dupes}')

In [ ]:
# Descriptive statistics for numerical columns
df.describe().round(2)

In [ ]:
# Categorical column summaries
cat_cols = ['origin_city', 'origin_state', 'destination_city', 'destination_state', 'truck_type']
for col in cat_cols:
    print(f'\n{col}: {df[col].nunique()} unique values')
    print(df[col].value_counts().head(5))

---
## 3. Exploratory Data Analysis (EDA)

### 3.1 Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
sns.histplot(df['actual_trip_cost_inr'], bins=50, kde=True, color='#ff6b35', ax=axes[0])
axes[0].set_title('Distribution of Actual Trip Cost (INR)', fontsize=14)
axes[0].set_xlabel('Trip Cost (₹)')
axes[0].axvline(df['actual_trip_cost_inr'].mean(), color='red', linestyle='--', label=f"Mean: ₹{df['actual_trip_cost_inr'].mean():,.0f}")
axes[0].axvline(df['actual_trip_cost_inr'].median(), color='blue', linestyle='--', label=f"Median: ₹{df['actual_trip_cost_inr'].median():,.0f}")
axes[0].legend()

# Box plot
sns.boxplot(y=df['actual_trip_cost_inr'], color='#ff6b35', ax=axes[1])
axes[1].set_title('Box Plot of Trip Cost', fontsize=14)
axes[1].set_ylabel('Trip Cost (₹)')

plt.tight_layout()
plt.show()

### 3.2 Cost by Truck Type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot by truck type
truck_order = df.groupby('truck_type')['actual_trip_cost_inr'].median().sort_values().index
sns.boxplot(x='truck_type', y='actual_trip_cost_inr', data=df, order=truck_order,
            palette='viridis', ax=axes[0])
axes[0].set_title('Trip Cost by Truck Type', fontsize=14)
axes[0].set_xlabel('Truck Type')
axes[0].set_ylabel('Trip Cost (₹)')

# Count plot
sns.countplot(x='truck_type', data=df, order=truck_order,
              palette='viridis', ax=axes[1])
axes[1].set_title('Trips per Truck Type', fontsize=14)
axes[1].set_xlabel('Truck Type')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

### 3.3 Distance vs. Trip Cost

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
scatter = ax.scatter(df['distance_km'], df['actual_trip_cost_inr'],
                     c=df['truck_type'].map({'LCV': 0, 'Medium': 1, 'Heavy': 2, 'Trailer': 3}),
                     alpha=0.4, s=10, cmap='viridis')
ax.set_xlabel('Distance (km)', fontsize=12)
ax.set_ylabel('Actual Trip Cost (₹)', fontsize=12)
ax.set_title('Distance vs. Trip Cost (coloured by Truck Type)', fontsize=14)

# Legend
import matplotlib.patches as mpatches
legend_labels = ['LCV', 'Medium', 'Heavy', 'Trailer']
colors = plt.cm.viridis(np.linspace(0, 1, 4))
patches = [mpatches.Patch(color=colors[i], label=legend_labels[i]) for i in range(4)]
ax.legend(handles=patches, title='Truck Type')

plt.tight_layout()
plt.show()

### 3.4 Correlation Heatmap (Numerical Features)

In [ ]:
# Select numerical columns only
num_cols = NUMERICAL_FEATURES + [TARGET_COLUMN]
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Heatmap — Numerical Features vs. Target', fontsize=14)
plt.tight_layout()
plt.show()

### 3.5 Top Origin & Destination Cities

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Origin cities
origin_counts = df['origin_city'].value_counts()
sns.barplot(x=origin_counts.values, y=origin_counts.index, palette='YlOrRd_r', ax=axes[0])
axes[0].set_title('Trips by Origin City', fontsize=14)
axes[0].set_xlabel('Count')

# Destination cities
dest_counts = df['destination_city'].value_counts()
sns.barplot(x=dest_counts.values, y=dest_counts.index, palette='YlOrRd_r', ax=axes[1])
axes[1].set_title('Trips by Destination City', fontsize=14)
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()

### 3.6 Monthly Trip Cost Trends

In [ ]:
monthly_avg = df.groupby('month')['actual_trip_cost_inr'].agg(['mean', 'median', 'count'])

fig, ax1 = plt.subplots(figsize=(12, 5))

month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

ax1.plot(monthly_avg.index, monthly_avg['mean'], 'o-', color='#ff6b35', label='Mean Cost', linewidth=2)
ax1.plot(monthly_avg.index, monthly_avg['median'], 's--', color='#4ecdc4', label='Median Cost', linewidth=2)
ax1.set_xlabel('Month')
ax1.set_ylabel('Trip Cost (₹)')
ax1.set_title('Monthly Average Trip Cost', fontsize=14)
ax1.set_xticks(range(1, 13))
ax1.set_xticklabels(month_names)
ax1.legend()
ax1.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 4. Feature/Target Separation & Leakage Prevention

In [ ]:
print('Columns EXCLUDED from model inputs (leakage / identifier / label):')
for col in DROP_COLUMNS:
    print(f'  ❌ {col}')

print(f'\nTarget: {TARGET_COLUMN}')
print(f'\nFeature columns ({len(ALL_FEATURE_COLUMNS)}):')
for col in ALL_FEATURE_COLUMNS:
    print(f'  ✅ {col}')

In [ ]:
# Separate features and target
X, y = separate_features_target(df)

print(f'Features shape: {X.shape}')
print(f'Target shape:   {y.shape}')
print(f'\nTarget (actual_trip_cost_inr) NOT in features: {TARGET_COLUMN not in X.columns}')
print(f'Revenue / Profit NOT in features: {"freight_revenue_inr" not in X.columns and "trip_profit_inr" not in X.columns}')

---
## 5. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)

print(f'Training set:  {X_train.shape[0]:,} samples ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Test set:      {X_test.shape[0]:,} samples ({X_test.shape[0]/len(X)*100:.0f}%)')

---
## 6. Preprocessing (Fitted on Training Data Only)

- **Numerical features** → `StandardScaler` (zero mean, unit variance)
- **Categorical features** → `OneHotEncoder` (handle_unknown='ignore')

The preprocessor is fitted **only on training data** to prevent data leakage.

In [ ]:
# Build and fit preprocessor on training data only
preprocessor = build_preprocessor()

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)  # transform only, NO fit

feature_names = get_feature_names(preprocessor)

print(f'Processed training features shape: {X_train_processed.shape}')
print(f'Processed test features shape:     {X_test_processed.shape}')
print(f'Total features after encoding:     {len(feature_names)}')
print(f'\nNumerical features: {len(NUMERICAL_FEATURES)}')
print(f'Categorical features: {len(CATEGORICAL_FEATURES)} → {X_train_processed.shape[1] - len(NUMERICAL_FEATURES)} one-hot columns')

---
## 7. Baseline Models

Before training the ANN, we establish baselines for comparison.

In [ ]:
def compute_metrics(y_true, y_pred, model_name):
    """Compute and display regression metrics."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f'\n{model_name}:')
    print(f'  MAE:  ₹{mae:>10,.2f}')
    print(f'  RMSE: ₹{rmse:>10,.2f}')
    print(f'  R²:    {r2:>10.4f}')
    return {'mae': float(mae), 'rmse': float(rmse), 'r2': float(r2)}

# 1. Dummy Regressor (mean)
baseline = build_baseline(strategy='mean')
baseline.fit(X_train_processed, y_train)
baseline_pred = baseline.predict(X_test_processed)
baseline_metrics = compute_metrics(y_test.values, baseline_pred, 'Baseline — DummyRegressor (Mean)')

# 2. Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train_processed, y_train)
lr_pred = lr_model.predict(X_test_processed)
lr_metrics = compute_metrics(y_test.values, lr_pred, 'Linear Regression')

---
## 8. ANN Model Architecture

Architecture: **Input → Dense(128, ReLU) → Dropout(0.2) → Dense(64, ReLU) → Dropout(0.2) → Dense(32, ReLU) → Dense(1, Linear)**

- **Loss:** Mean Squared Error (MSE)
- **Optimizer:** Adam (lr=0.001)
- **Early Stopping:** Monitors `val_loss`, patience=15, restores best weights

In [ ]:
input_dim = X_train_processed.shape[1]

model = keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(128, activation='relu', name='hidden_1'),
    layers.Dropout(0.2),
    layers.Dense(64, activation='relu', name='hidden_2'),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu', name='hidden_3'),
    layers.Dense(1, activation='linear', name='output'),
], name='TruckCostANN')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae'],
)

model.summary()

---
## 9. Model Training

In [ ]:
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1,
)

history = model.fit(
    X_train_processed, y_train.values,
    validation_split=0.15,
    epochs=150,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1,
)

---
## 10. Training & Validation Loss Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_title('Training & Validation Loss (MSE)', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE
axes[1].plot(history.history['mae'], label='Train MAE', linewidth=2)
axes[1].plot(history.history['val_mae'], label='Val MAE', linewidth=2)
axes[1].set_title('Training & Validation MAE', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE (₹)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 11. Test Set Evaluation

In [ ]:
y_pred = model.predict(X_test_processed, verbose=0).flatten()

ann_metrics = compute_metrics(y_test.values, y_pred, 'ANN (Test Set)')

# Comparison table
comparison = pd.DataFrame({
    'Model': ['DummyRegressor (Mean)', 'Linear Regression', 'ANN'],
    'MAE (₹)': [baseline_metrics['mae'], lr_metrics['mae'], ann_metrics['mae']],
    'RMSE (₹)': [baseline_metrics['rmse'], lr_metrics['rmse'], ann_metrics['rmse']],
    'R²': [baseline_metrics['r2'], lr_metrics['r2'], ann_metrics['r2']],
})
comparison.style.format({'MAE (₹)': '₹{:,.2f}', 'RMSE (₹)': '₹{:,.2f}', 'R²': '{:.4f}'}).set_caption('Model Comparison')

---
## 12. Actual vs. Predicted & Residual Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Actual vs Predicted scatter
axes[0].scatter(y_test.values, y_pred, alpha=0.3, s=10, color='steelblue')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Ideal (y=x)')
axes[0].set_xlabel('Actual Trip Cost (₹)', fontsize=12)
axes[0].set_ylabel('Predicted Trip Cost (₹)', fontsize=12)
axes[0].set_title('Actual vs. Predicted Trip Cost', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Residual plot
residuals = y_test.values - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.3, s=10, color='coral')
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_xlabel('Predicted Trip Cost (₹)', fontsize=12)
axes[1].set_ylabel('Residual (₹)', fontsize=12)
axes[1].set_title('Residual Analysis', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Residual distribution
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(residuals, bins=50, kde=True, color='coral', ax=ax)
ax.set_title('Residual Distribution', fontsize=14)
ax.set_xlabel('Residual (₹)')
ax.axvline(0, color='black', linewidth=1, linestyle='--')
plt.tight_layout()
plt.show()

print(f'Mean residual: ₹{residuals.mean():,.2f}')
print(f'Std of residuals: ₹{residuals.std():,.2f}')

---
## 13. Save Trained Artefacts

Saving:
1. Keras model (`truck_cost_ann.keras`)
2. Preprocessing pipeline (`preprocessor.pkl`)
3. Feature schema (`feature_schema.json`)
4. Evaluation metrics (`evaluation_metrics.json`)
5. Baseline Linear Regression (`baseline_lr.pkl`)

In [ ]:
MODEL_DIR = os.path.join('..', 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# 1. Keras model
model_path = os.path.join(MODEL_DIR, 'truck_cost_ann.keras')
model.save(model_path)
print(f'✅ ANN model saved to {model_path}')

# 2. Preprocessor
preprocessor_path = os.path.join(MODEL_DIR, 'preprocessor.pkl')
joblib.dump(preprocessor, preprocessor_path)
print(f'✅ Preprocessor saved to {preprocessor_path}')

# 3. Feature schema
schema = {
    'numerical_features': NUMERICAL_FEATURES,
    'categorical_features': CATEGORICAL_FEATURES,
    'all_feature_columns': ALL_FEATURE_COLUMNS,
    'target': TARGET_COLUMN,
    'processed_feature_names': feature_names,
    'input_dim': int(X_train_processed.shape[1]),
}
schema_path = os.path.join(MODEL_DIR, 'feature_schema.json')
with open(schema_path, 'w') as f:
    json.dump(schema, f, indent=2)
print(f'✅ Feature schema saved to {schema_path}')

# 4. Metrics
all_metrics = {
    'baseline_dummy_mean': baseline_metrics,
    'baseline_linear_regression': lr_metrics,
    'ann': ann_metrics,
    'note': (
        'This dataset is synthetic (formula-generated). High R² values '
        'reflect the deterministic formula, not real-world prediction '
        'accuracy. Do not quote these metrics as production performance.'
    ),
}
metrics_path = os.path.join(MODEL_DIR, 'evaluation_metrics.json')
with open(metrics_path, 'w') as f:
    json.dump(all_metrics, f, indent=2)
print(f'✅ Evaluation metrics saved to {metrics_path}')

# 5. Baseline LR
lr_path = os.path.join(MODEL_DIR, 'baseline_lr.pkl')
joblib.dump(lr_model, lr_path)
print(f'✅ Baseline LR saved to {lr_path}')

# Training history plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[1].plot(history.history['mae'], label='Train MAE')
axes[1].plot(history.history['val_mae'], label='Val MAE')
axes[1].set_title('MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'training_history.png'), dpi=150)
plt.show()

# Actual vs Predicted plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_test.values, y_pred, alpha=0.3, s=10, color='steelblue')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, 'r--', linewidth=1.5)
axes[0].set_title('Actual vs Predicted')
axes[0].grid(True, alpha=0.3)
residuals = y_test.values - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.3, s=10, color='coral')
axes[1].axhline(0, color='black')
axes[1].set_title('Residuals')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'actual_vs_predicted.png'), dpi=150)
plt.show()

print('\n🎉 All artefacts saved successfully!')

---
## 14. Findings, Limitations & Future Work

### Key Findings
1. The ANN model significantly outperforms the mean-baseline DummyRegressor.
2. Both Linear Regression and ANN achieve high R² on this dataset — expected because the synthetic target was generated from a deterministic formula involving the input features.
3. Distance, truck capacity, route factor, and traffic factor have strong correlations with trip cost.
4. The ANN architecture with 128→64→32 hidden units with dropout regularisation converges well with early stopping.

### ⚠️ Limitations
1. **Synthetic Data:** The target (`actual_trip_cost_inr`) was formula-generated. High evaluation metrics reflect this formula, not real-world prediction accuracy.
2. **No Real Validation:** This model has not been validated against actual truck operating cost data.
3. **Limited Feature Engineering:** No derived features (e.g., cost-per-km, load-utilisation ratio) were explored.
4. **Static Diesel Prices:** The model treats diesel price as a static input; real-world prices fluctuate dynamically.
5. **No Temporal Patterns:** Month is treated as a numerical feature; cyclical encoding could improve seasonality capture.

### 🚀 Future Improvements
1. Collect and validate against real-world truck trip cost data.
2. Add feature engineering (cost ratios, load-utilisation, cyclical month encoding).
3. Try hyperparameter tuning (learning rate, layer sizes, dropout rate).
4. Explore tree-based models (XGBoost, LightGBM) for comparison.
5. Add SHAP or LIME for model interpretability.
6. Build an API endpoint for batch predictions.
7. Implement CI/CD pipeline for model retraining.